In [1]:
pip install unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.1 MB/s eta 0:00:0031m14.3 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.1 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 30.2 MB/s eta 0:00:0031m30.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.9 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.0 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
import pandas as pd 
import unstructured 

In [ ]:
import json
import requests
from requests.auth import HTTPBasicAuth
# # Hardcoded JSON file path
# with open('API_CONFLUENCE.json', 'r') as file:
#     confluence_config = json.load(file)
# Parse the PAGE_URL to extract page_id
parsed_url = urlparse(confluence_config['PAGE_URL'])
path_parts = parsed_url.path.split('/')
page_id = path_parts[-2] if len(path_parts) >= 2 else None
# Confluence API endpoints
API_URL = f"{confluence_config['BASE_URL']}/rest/api/content"
BASIC_AUTH = HTTPBasicAuth(confluence_config['EMAIL'], confluence_config['TOKEN'])

# Confluence authentication headers
headers = {
    "Content-Type": "application/json",
    "Authorization": f"{confluence_config['AUTH_METHOD']} {confluence_config['TOKEN']}"
}
# Fetch page details
response = requests.get(f"{API_URL}/{page_id}?expand=version,body.view,metadata.labels", headers=headers, auth=BASIC_AUTH)
# Check for success
if response.status_code == 200:
    page_data = response.json()
    # Extract label from the response
    page_labels = page_data.get('metadata', {}).get('labels', {}).get('results', [])
    label_values = [page.get('label', None) for page in page_labels] if page_labels else None
     # Construct page details dictionary
    page_details_dict = {
        "page_title": page_data.get('title', 'No Title'),
        "page_author": page_data.get('version', {}).get('by', {}).get('publicName', None),
        "page_content": page_data.get('body', {}).get('view', {}).get('value', ''),
        "page_label": label_values,
    }
    # Save page details to a JSON file
    output_file_path = 'page_details.json'
    with open(output_file_path, 'w') as output_file:
        json.dump(page_details_dict, output_file, indent=2)
    print(f"Page details saved to {output_file_path}")
    # Print page details
    print("Page Details:")
    for key, value in page_details_dict.items():
        print(f"{key}: {value}")
else:
    print(f"Failed to fetch page details. Status code: {response.status_code}")
    print(response.text)

In [3]:
# headings to look for :
# Problem
# Solution & key reason
# Context
# Options Decision Criteria and Analysis (sometimes different, sometimes in the same heading ) 

# give a Unique ID, store it at time of creation against page_name for later verification


In [4]:
def read_confluence_page(url, email, token):

    # authenticate
    auth = (email, token)

    # fetch page content
    response = requets.get(url, auth=auth)
    response.raise_for_status()
    
    # Read the Confluence page
    page = unstructured.unstructured.partition_file(response.text)
    # Initialize a dictionary to store data     
    data = {} 
    # Iterate over the subsections     
    for subsection in page.subsections:
        subsection_heading = subsection.heading.strip() 
        subsection_text = subsection.text.strip()
        # Check if the subsection contains a table 
        if subsection.tables:
            # Convert the first table to a DataFrame             
            table_df = pd.DataFrame(subsection.tables[0])
            data[subsection_heading] = table_df
        else:             
            data[subsection_heading] = subsection_text     
            
    # Create a DataFrame from the dictionary     
    df = pd.DataFrame(data)     
    return df
